# 03 — Named Entity Recognition (NER)
This notebook explores how ClauseGuard uses spaCy's pretrained NER model to automatically extract key entities from a contract — parties, dates, monetary amounts, and locations.

**Production file:** `backend/ner.py`

In [ ]:
import spacy
from IPython.display import display, HTML

nlp = spacy.load("en_core_web_sm")
print("spaCy model:", nlp.meta['name'], "| version:", nlp.meta['version'])

## 1. Run spaCy NER on a Sample Contract Segment

In [ ]:
sample_text = """This Services Agreement is entered into on June 25, 2026, by and between Acme Corp, located in New York, NY, and John Doe. The total compensation for the project is $5,000."""
doc = nlp(sample_text)

print(f"Total entities detected: {len(doc.ents)}\n")
print(f"{'Entity Text':<40} {'Label':<12} {'Explanation'}")
print("-" * 70)
for ent in doc.ents:
    print(f"{ent.text[:38]:<40} {ent.label_:<12} {spacy.explain(ent.label_)}")

## 2. Map spaCy labels to our simplified categories

In [ ]:
# spaCy has 18 entity types — we only care about 5
LABEL_MAP = {
    "PERSON": "parties",
    "ORG":    "parties",
    "DATE":   "dates",
    "MONEY":  "amounts",
    "GPE":    "locations",   # GPE = Geo-Political Entity
    "LOC":    "locations",
}

def extract_entities(text):
    doc = nlp(text)
    entities = {"parties": [], "dates": [], "amounts": [], "locations": []}
    for ent in doc.ents:
        category = LABEL_MAP.get(ent.label_)
        if category and ent.text not in entities[category]:
            entities[category].append(ent.text)
    return entities

entities = extract_entities(sample_text)
for category, items in entities.items():
    print(f"\n{category.upper()}:")
    for item in items:
        print(f"  • {item}")